# 02 - Tipo de Cambio 
Se enriquece la información de operaciones con datos externos: Tipo de cambio diario, Cotizaciones de mercado

In [0]:
import requests
import pandas as pd
from pyspark.sql.functions import col, when

In [0]:
df_ops = spark.table("bronze_byma.operaciones_raw")

In [0]:
url = "https://api.argentinadatos.com/v1/cotizaciones/dolares"

response = requests.get(url)
data = response.json()

df_dolar = pd.DataFrame(data)

In [0]:
df_dolar = df_dolar[
    df_dolar["casa"].str.lower() == "contadoconliqui"
]

#Solo fechas del dataset
df_dolar = df_dolar[
    (df_dolar["fecha"] >= "2026-01-01") &
    (df_dolar["fecha"] <= "2026-03-31")
]

In [0]:
df_dolar = df_dolar.rename(columns={
    "venta": "ccl"
})

df_dolar = df_dolar[["fecha", "ccl"]]

In [0]:
df_fecha_dolar = spark.createDataFrame(df_dolar)

In [0]:
df_fecha_dolar.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_byma.fecha_dolar")

In [0]:
df_tipo_cambio = spark.createDataFrame(df_dolar)

In [0]:
df_ops = df_ops.join(
    df_tipo_cambio,
    df_ops["fecha_particion"] == df_tipo_cambio["fecha"],
    "left"
)

In [0]:
df_ops = df_ops.withColumn(
    "precio_operado_usd",
    when(col("moneda") == "ARS", col("precio") / col("ccl"))
    .otherwise(col("precio"))
)

In [0]:
df_final = df_ops.select(
    col("id_transaccion"),
    col("id_cliente"),
    col("simbolo_titulo"),
    col("descripcion_titulo"),
    col("tipoTran"),
    col("moneda"),
    col("cantidad"),
    col("precio"),
    col("fecha_particion").alias("fecha"),
    col("origen"),
    col("precio_operado_usd")
)

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS silver_byma")

In [0]:
df_final.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_byma.tipo_cambio")